In [ ]:
import logging
import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm

from vasae.data.dataset import GPT2LayerActivations



logging.basicConfig(
    format="[%(levelname)s] %(asctime)s %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger()

In [ ]:
class CFG:
    seed = 42

    meta_path = "/scratch/b5bq/pu22650.b5bq/activations_gpt2_Geralt-Targaryen_openwebtext2/meta.json"
    layer_name = "transformer.h.5"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "gpt2"
    save_dir = "out"



def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
args = CFG()
set_seed(args.seed)
dataset = GPT2LayerActivations(args.meta_path, args.layer_name)


# box
不同样本同一层不同位置的box
- 中间近似正太，但重尾。外点只是因为不满足箱型图规则。
- activations大部分样本每个维度还是比较理想的，seq0确实可能大量数据更集中一些，但并不应该影响学习，数据量足够大，真的就是有几个噪声，应该也不影响。

In [ ]:
num_dims = 10    
fig, axs = plt.subplots(nrows=num_dims,ncols=1,figsize=(3.5, 25),layout="constrained")
for d in range(num_dims):
    x = dataset[0:-1][:,:,d]
    axs[d].boxplot(x, showfliers=True)
plt.title("layer 5 activations seq pos from 0 to 8 at dim 0")
plt.xlabel("Seq_i")
plt.ylabel("Value")
plt.show()

# center后
明显mean就是0了，应该会更好学

In [ ]:
dataset_c = GPT2LayerActivations(args.meta_path, args.layer_name,use_centralize=True)

In [ ]:
num_dims = 10    
fig, axs = plt.subplots(nrows=num_dims,ncols=1,figsize=(3.5, 25),layout="constrained")
for d in range(num_dims):
    x = dataset_c[0:-1][:,:,d]
    axs[d].boxplot(x, showfliers=True)
plt.xlabel("Seq_i")
plt.ylabel("Value")
plt.show()

# (seq,dim,mean)

In [ ]:
x = dataset[0:-1].mean(dim=0)
xmin = x.min().item()
xmax = x.max().item()
print(xmin, xmax)

In [ ]:
plt.boxplot(x.flatten(), showfliers=False)

In [ ]:
x = dataset[0:-1].mean(dim=0)
xmin = x.min().item()
xmax = x.max().item()



plt.figure()
plt.imshow(
    x.numpy(),
    aspect='auto',
    norm=SymLogNorm(
        linthresh=1,   # -20~20 线性
        linscale=5.0,   # 线性区在颜色轴上占更大比例
        vmin=xmin,
        vmax=xmax
    )
)
plt.xlabel("Dim (768)")
plt.ylabel("Seq (64)")
plt.colorbar()
plt.show()

In [ ]:
x = dataset[0:-1].std(dim=0)
xmin = x.min().item()
xmax = x.max().item()
print(xmin, xmax)

In [ ]:
plt.boxplot(x.flatten(), showfliers=False)

In [ ]:



plt.figure()
plt.imshow(
    x.numpy(),
    aspect='auto',

)
plt.xlabel("Dim (768)")
plt.ylabel("Seq (64)")
plt.colorbar()
plt.show()

In [ ]:


# 假设已有 tensor x，形状 [64, 768]
# x = your_tensor
x = dataset[0]

xmin = x.min().item()
xmax = x.max().item()

plt.figure()
plt.imshow(
    x.cpu().numpy(),
    aspect='auto',
    vmin=-2,
    vmax=2
)
plt.colorbar()
plt.xlabel("Dim (768)")
plt.ylabel("Seq (64)")
plt.show()

print("min:", xmin, "max:", xmax)

x.flatten().topk(10)

In [ ]:
x.std()

In [ ]:
import torch
x = dataset[0]
# x: [batch, seq, hidden] 或任意 shape
abs_x = x.flatten()

p99   = torch.quantile(abs_x, 0.9)
p999  = torch.quantile(abs_x, 0.999)

print(p99.item(), p999.item())

In [ ]:
import torch
x = torch.tensor([-1,2,3], dtype=torch.float32)
torch.quantile(x, 0.5)

# heatmap

In [ ]:
from matplotlib.colors import SymLogNorm

x = dataset[0]

xmin = x.min().item()
xmax = x.max().item()


plt.figure()
plt.imshow(
    x.numpy(),
    aspect='auto',
    norm=SymLogNorm(
        linthresh=4,   # -20~20 线性
        linscale=5.0,   # 线性区在颜色轴上占更大比例
        vmin=xmin,
        vmax=xmax
    )
)
plt.colorbar()
plt.show()

# centralized后箱型图

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

class AsymPieceLogNorm(Normalize):
    """
    [-5, 4] 线性放大；左侧/右侧对数压缩
    输出严格落在 [0,1]，colorbar 不会乱
    """
    def __init__(self, vmin, vmax, lin_min, lin_max,
                 lin_frac=0.6, eps=1e-6):
        super().__init__(vmin=vmin, vmax=vmax, clip=True)
        self.lin_min = lin_min
        self.lin_max = lin_max
        self.eps = eps
        self.left_frac = (1 - lin_frac) / 2
        self.mid_frac  = lin_frac
        self.right_frac = (1 - lin_frac) / 2

    def __call__(self, value, clip=None):
        v = np.asarray(value, dtype=float)
        v = np.clip(v, self.vmin, self.vmax)

        out = np.empty_like(v, dtype=float)

        L = self.left_frac
        M = self.mid_frac
        # R = self.right_frac  # 其实= L

        # 左侧：vmin -> 0, lin_min -> L（对数压缩）
        mask_l = v < self.lin_min
        if np.any(mask_l):
            ratio = (v[mask_l] - self.vmin) / (self.lin_min - self.vmin)
            ratio = np.clip(ratio, self.eps, 1.0)
            t = np.log10(ratio) / np.log10(self.eps)   # eps->1, 1->0
            out[mask_l] = L * (1 - t)                  # vmin≈0, lin_min=L

        # 中间：lin_min -> L, lin_max -> L+M（线性放大）
        mask_m = (v >= self.lin_min) & (v <= self.lin_max)
        out[mask_m] = L + M * (v[mask_m] - self.lin_min) / (self.lin_max - self.lin_min)

        # 右侧：lin_max -> L+M, vmax -> 1（对数压缩）
        mask_r = v > self.lin_max
        if np.any(mask_r):
            ratio = (v[mask_r] - self.lin_max) / (self.vmax - self.lin_max)
            ratio = np.clip(ratio, self.eps, 1.0)
            t = np.log10(ratio) / np.log10(self.eps)   # eps->1, 1->0
            out[mask_r] = (L + M) + L * (1 - t)        # lin_max=L+M, vmax=1

        return out

# ---------------- 用法 ----------------
# x: torch.Tensor [64,768]
# x = your_tensor.cpu().numpy()
x = np.random.randn(64, 768) * 20
x = np.clip(x, -250, 2000)

norm = AsymPieceLogNorm(vmin=-250, vmax=2000, lin_min=-5, lin_max=4, lin_frac=0.3)

fig, ax = plt.subplots()
im = ax.imshow(x, aspect="auto", norm=norm)

# 关键：手动指定刻度，避免自动挤爆
ticks = [-250, -60, -5, 0, 4, 10, 100, 500, 2000]
cbar = fig.colorbar(im, ax=ax, ticks=ticks)
cbar.ax.set_yticklabels([str(t) for t in ticks])

plt.show()

In [ ]:
# emblayer 可视化